In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
!pip install dataretrieval
!pip install pytorch_forecasting
from wavelet_class import *
from preprocess_data import *
from transformer_knn import *
from empirical_copula import EmpiricalCopulaSimulator
from marginals_pos import *
from KNN_NS import *
from trajectories import *
from gauge_check import *
from joint_analog import *
from analog_quantile_sampler import AnalogQuantileSampler
import requests_cache
requests_cache.install_cache("nwis_cache", expire_after=7*24*3600)
import matplotlib as mpl
from sklearn.preprocessing import StandardScaler
from scipy.stats import pearsonr, spearmanr
from sklearn.decomposition import PCA
from matplotlib.colors import LinearSegmentedColormap
import os
import sys
from contextlib import contextmanager

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pytorch_forecasting]torch_forecasting]


In [2]:
mpl.rcParams.update({
    "font.family":       "sans-serif",
    "font.sans-serif":   ["Inter", "Helvetica Neue", "Arial", "DejaVu Sans"],
    "font.size":         11,
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "axes.grid":         True,
    "grid.color":        "#e5e5e5",
    "grid.linewidth":    0.6,
    "xtick.labelsize":   9,
    "ytick.labelsize":   9,
    "axes.labelsize":    10,
    "axes.titlesize":    11,
    "axes.titleweight":  "normal",
    "figure.dpi":        150,
})

In [3]:
def assemble_and_export(
    expanded,
    resampled_all,
    ds,
    sites,
    thr_map,
    parent_dir,
    run_name,
    apply_threshold_mask=False,
    export_results=False,
):
    """
    Shared post-simulation assembly used by both joint-analog and copula modes.
    Produces events_export (true events only) and monthly_sim (full monthly table).
    """
    # ── 1) Event-level melt ───────────────────────────────────────────────────
    export_sim = events_to_export(
        expanded,
        signal_suffix="forecast_signal",
        ds=ds,
        keep_zeros=True,
    )

    # ── 2) Threshold mask ─────────────────────────────────────────────────────
    thr_s = pd.Series(thr_map, name="thr")
    export_sim = export_sim.join(thr_s, on="Site")

    orig_freq      = export_sim["Frequency"].copy()
    orig_intensity = export_sim["Intensity"].copy()

    if apply_threshold_mask:
        below_pos_int     = (orig_intensity > 0) & (orig_intensity < export_sim["thr"])
        freq_changed_mask = below_pos_int & (orig_freq > 0)
        sites_order       = sorted(thr_map.keys())

        freq_changed_counts = (
            export_sim.loc[freq_changed_mask]
            .groupby("Site").size()
            .reindex(sites_order, fill_value=0)
        )
        int_changed_counts = (
            export_sim.loc[below_pos_int]
            .groupby("Site").size()
            .reindex(sites_order, fill_value=0)
        )

        print("Events zeroed (freq > 0, intensity sub-threshold) by site:")
        for site, cnt in freq_changed_counts.items():
            print(f"  {site}: {cnt}")
        print("Events zeroed (intensity > 0, sub-threshold) by site:")
        for site, cnt in int_changed_counts.items():
            print(f"  {site}: {cnt}")

        export_sim.loc[below_pos_int, ["Frequency", "Intensity", "Duration"]] = 0.0

    export_sim             = export_sim.drop(columns="thr")
    export_sim_for_monthly = export_sim.copy()

    # ── 3) True events only ───────────────────────────────────────────────────
    events_export = export_sim[
        (export_sim["Frequency"] > 0) &
        (export_sim["Intensity"] > 0)
    ].copy()

    # ── 4) Base monthly table ─────────────────────────────────────────────────
    base_period = ds.monthly_max.index[-1].to_period("M") + 1
    base_frames = []

    for s in sites:
        freq_col = f"{s}_frequency"
        sig_col  = f"{s}_forecast_signal"

        if (freq_col not in resampled_all.columns) or (sig_col not in resampled_all.columns):
            continue

        tmp = resampled_all[[freq_col, sig_col]].copy()
        tmp = tmp.rename(columns={freq_col: "Frequency", sig_col: "Signal"})
        tmp["Site"] = s
        tmp = tmp.reset_index().rename(columns={"sim_id": "Sim"})

        offsets = tmp["t"].astype(int) - 1
        periods = pd.PeriodIndex(
            [base_period + int(k) for k in offsets.to_numpy()], freq="M"
        )
        tmp["MonthPeriod"] = periods
        tmp["Month"]       = periods.to_timestamp("M")
        base_frames.append(tmp[["Sim", "Site", "Month", "MonthPeriod", "Signal", "Frequency"]])

    base_monthly = pd.concat(base_frames, ignore_index=True)
    base_monthly["Frequency"] = base_monthly["Frequency"].fillna(0.0)

    # ── 5) Aggregate event intensity/duration per month ───────────────────────
    export_sim_for_monthly["Month"]       = pd.to_datetime(export_sim_for_monthly["Month"])
    export_sim_for_monthly["MonthPeriod"] = export_sim_for_monthly["Month"].dt.to_period("M")

    event_monthly = (
        export_sim_for_monthly
        .groupby(["Sim", "Site", "MonthPeriod"], as_index=False)
        .agg({"Intensity": "max", "Duration": "max"})
    )

    # ── 6) Merge -> monthly_sim ───────────────────────────────────────────────
    monthly_sim = base_monthly.merge(
        event_monthly, on=["Sim", "Site", "MonthPeriod"], how="left"
    )
    monthly_sim["Month"]     = monthly_sim["MonthPeriod"].dt.to_timestamp("M")
    monthly_sim["Intensity"] = monthly_sim["Intensity"].fillna(0.0)
    monthly_sim["Duration"]  = monthly_sim["Duration"].fillna(0.0)
    monthly_sim              = monthly_sim.drop(columns=["MonthPeriod"])
    monthly_sim              = monthly_sim[
        ["Sim", "Site", "Month", "Signal", "Frequency", "Intensity", "Duration"]
    ]

    # ── 7) Write outputs ──────────────────────────────────────────────────────
    if export_results:
        events_export.to_csv(f"{parent_dir}/events_simulated_{run_name}.csv", index=False)
        monthly_sim.to_csv(  f"{parent_dir}/fids_simulated_{run_name}.csv",   index=False)
        print(f"Saved events_simulated_{run_name}.csv and fids_simulated_{run_name}.csv")

    return events_export, monthly_sim

In [4]:
@contextmanager
def suppress_output():
    with open(os.devnull, 'w') as devnull:
        old_stdout = sys.stdout
        sys.stdout = devnull
        try:
            yield
        finally:
            sys.stdout = old_stdout

In [5]:
def compute_corr_results(resampled_all, climate_filtered, sites,
                         suffixes, ja_sampler=None):

    def tail_corr_from_wide(wide, q=0.9):
        X    = wide.copy()
        q_hi = X.quantile(q)
        q_lo = X.quantile(1 - q)
        S    = pd.DataFrame(0.0, index=X.index, columns=X.columns)
        for col in X.columns:
            S.loc[X[col] > q_hi[col], col] =  1.0
            S.loc[X[col] < q_lo[col], col] = -1.0
        return S.corr(method="pearson")

    def build_wide(df, sites, suffix):
        cols = [f"{s}{suffix}" for s in sites if f"{s}{suffix}" in df.columns]
        return df[cols].copy()

    # ── Step 1: Pearson correlations ──────────────────────────────────────────
    corr_results = {}
    for suffix in suffixes:
        if ja_sampler is not None:
            obs_c, sim_c = ja_sampler.spatial_correlation_summary(
                resampled_all, climate_filtered, var_suffix=suffix
            )
        else:
            obs_wide = build_wide(climate_filtered, sites, suffix).dropna()
            sim_wide = (
                build_wide(resampled_all, sites, suffix)
                .reset_index(level="sim_id", drop=True)
                .dropna()
            )
            obs_c = obs_wide.corr(method="pearson", min_periods=20)
            sim_c = sim_wide.corr(method="pearson", min_periods=20)

        corr_results[suffix] = (obs_c, sim_c)

    # ── Step 2: Tail correlations — identical path for both modes ─────────────
    tail_results = {}
    for suffix in suffixes:
        obs_c, sim_c = corr_results[suffix]
        obs_wide = build_wide(climate_filtered, sites, suffix)
        sim_wide = (
            build_wide(resampled_all, sites, suffix)
            .reset_index(level="sim_id", drop=True)
        )
        obs_tail   = tail_corr_from_wide(obs_wide.dropna())
        sim_tail   = tail_corr_from_wide(sim_wide.dropna())
        site_order = obs_c.columns.tolist()
        obs_tail   = obs_tail.reindex(index=site_order, columns=site_order)
        sim_tail   = sim_tail.reindex(index=site_order, columns=site_order)
        tail_results[suffix] = (obs_tail, sim_tail)

    return corr_results, tail_results

In [6]:
def plot_spatial_correlations(corr_results, tail_results, run_name):
    """
    Shared correlation plot — works for both joint-analog and copula mode.
    Layout: 3 rows x 4 cols
      Row 0: Pearson heatmap obs/sim + Tail heatmap obs/sim  (intensity)
      Row 1: Intensity + Frequency scatter (Pearson cols 0-1, Tail cols 2-3)
      Row 2: Duration  + Signal    scatter (Pearson cols 0-1, Tail cols 2-3)
    """
    muted_blue = LinearSegmentedColormap.from_list(
        "muted_blue", ["#F5F5F7", "#6B82A8", "#2E4A72"]
    )
    muted_red = LinearSegmentedColormap.from_list(
        "muted_red", ["#F5F5F7", "#C97070", "#A32D2D"]
    )

    row_letters = [
        ["a)", "b)", "c)", "d)"],
        ["e)", "f)", "g)", "h)"],
        ["i)", "j)", "k)", "l)"],
    ]

    def plabel(ax, letter):
        ax.text(-0.1, 1.2, letter, transform=ax.transAxes,
                fontsize=11, fontweight="bold", va="top")

    def corr_scatter(ax, obs_c, sim_c, title, color="#5F77A2"):
        if obs_c.empty or sim_c.empty:
            ax.set_title(f"{title} (no data)", fontsize=11)
            return
        common = obs_c.columns.intersection(sim_c.columns)
        obs_c  = obs_c.loc[common, common]
        sim_c  = sim_c.loc[common, common]
        tri   = np.triu_indices(len(obs_c), k=1)
        r_obs = obs_c.values[tri]
        r_sim = sim_c.values[tri]
        mask  = np.isfinite(r_obs) & np.isfinite(r_sim)
        ax.scatter(r_obs[mask], r_sim[mask],
                   alpha=0.15, s=8, color=color, linewidths=0)
        ax.plot([-1, 1], [-1, 1], color="#444", lw=0.8, linestyle="--")
        ax.set_xlabel("Observed r", fontsize=9)
        ax.set_ylabel("Simulated r", fontsize=9)
        ax.set_title(title, fontsize=11)
        ax.set_xlim(-0.5, 1)
        ax.set_ylim(-0.5, 1)
        ax.grid(color="#e5e5e5", linewidth=0.5, zorder=0)
        ax.set_axisbelow(True)

    fig = plt.figure(figsize=(10, 5.5), constrained_layout=True)
    gs  = fig.add_gridspec(3, 4, height_ratios=[1.4, 1, 1])

    # ── Row 0: heatmaps ───────────────────────────────────────────────────────
    obs_corr,   sim_corr   = corr_results["_max_intensity"]
    obs_tail_h, sim_tail_h = tail_results["_max_intensity"]

    heatmap_data = [
        (obs_corr,   muted_blue, 0, 1, "a)", "Observed"),
        (sim_corr,   muted_blue, 0, 1, "b)", "Simulated"),
        (obs_tail_h, muted_red,  0, 1, "c)", "Observed"),
        (sim_tail_h, muted_red,  0, 1, "d)", "Simulated"),
    ]
    for col_idx, (data, cmap, vmin, vmax, letter, title) in enumerate(heatmap_data):
        ax = fig.add_subplot(gs[0, col_idx])
        ax.imshow(data.values, vmin=vmin, vmax=vmax, cmap=cmap)
        ax.set_title(title, fontsize=11)
        ax.set_xticks([]); ax.set_yticks([])
        fig.colorbar(ax.images[0], ax=ax, fraction=0.046, pad=0.04)
        plabel(ax, letter)

    # ── Row 1: Intensity + Frequency scatter ──────────────────────────────────
    row1_cfg = [
        (0, corr_results["_max_intensity"], "Intensity", "#5F77A2"),
        (1, corr_results["_frequency"],     "Frequency", "#5F77A2"),
        (2, tail_results["_max_intensity"], "Intensity", "#A32D2D"),
        (3, tail_results["_frequency"],     "Frequency", "#A32D2D"),
    ]
    for col_idx, (_, result, label, color) in enumerate(row1_cfg):
        ax = fig.add_subplot(gs[1, col_idx])
        corr_scatter(ax, *result, label, color=color)
        plabel(ax, row_letters[1][col_idx])

    # ── Row 2: Duration + Signal scatter ──────────────────────────────────────
    row2_cfg = [
        (0, corr_results["_duration_days"], "Duration", "#5F77A2"),
        (1, corr_results["_max_z_wave"],    "Signal",   "#5F77A2"),
        (2, tail_results["_duration_days"], "Duration", "#A32D2D"),
        (3, tail_results["_max_z_wave"],    "Signal",   "#A32D2D"),
    ]
    for col_idx, (_, result, label, color) in enumerate(row2_cfg):
        ax = fig.add_subplot(gs[2, col_idx])
        corr_scatter(ax, *result, label, color=color)
        plabel(ax, row_letters[2][col_idx])

    fig.text(0.27, 1.01, "Pairwise Pearson Correlation",
             ha="center", fontsize=12, fontweight="500")
    fig.text(0.74, 1.01, "Pairwise Tail Correlation (q>0.9)",
             ha="center", fontsize=12, fontweight="500")

    #plt.savefig(f"tail_spatial_correlations_{run_name}.png",
    #            dpi=500, bbox_inches="tight")
    plt.show()

# PARAMS

In [7]:
# ── General ───────────────────────────────────────────────────────────────
n_sims           = 1000          # number of simulations
forecast_horizon = 60            # months to forecast (default 60 = 5 years)
verbose = False                   # False to suppress all printed output

# ── Data Load ─────────────────────────────────────────────────────────────
experiment       = 'default'     # 'default' | 'subset'
std_thres        = 'bimonthly'   # threshold method for event extraction
climate_vars     = ['nao', 'pdo', 'enso', 'gta']
discharge_source = 'local'       # 'local' | 'nwis'
plot_gauges      = False

# ── Wavelet ───────────────────────────────────────────────────────────────
use_wave         = True          # True | False: wavelet-filter signal or use raw monthly maxima
siglvl           = 0.95          # significance level (standard 0.95)
sigtest          = 'default'     # 'default' (red noise) | 'white'
plot_wave        = False

# ── Transformer ───────────────────────────────────────────────────────────
seq_length       = 120
hidden_dim       = 64
num_layers       = 2
lr               = 0.001
epochs           = 50
batch_size       = 64
patience         = 10
early_stop       = True
temp             = 1.0
alpha            = 1.0
beta             = 1.0
val_frac         = 0.1
test_frac        = 0
model_kind       = 'autoformer'  # 'autoformer' | 'transformer'
loss_func        = 'mse'         # 'mse' | 'w_mse' | 'expectile' | 'quantile'
mode             = 'analog_mode' # 'analog_mode' | 'forecast_mode'
assess_pred      = False
plot_transformer = False
check_PC         = False
plot_PC          = False

# ── NS Sampler & Empirical copula ────────────────────────────────────────────────────────────
simulation_mode  = 'copula' # 'joint-analog' | 'copula'
joint_K          = None     # None = auto sqrt(n_windows)
joint_temp       = 1.0
joint_seed       = 0
aq_std           = 0.02  # default 0.02, 0.0 = deterministic, 0.05 = more stochastic
signal_dist      = 'logspline' # gamma recommended for parametric
intensity_dist   = 'logspline' # exponential_t recommended for parametric
duration_dist    = 'logspline' # lognormal recommended for parametric
frequency_dist   = 'logspline' # neg_bin recommended for parametric
plot_marginals   = False # empirical copula marginals
cop_plot_qq      = False # empirical copula qq
plot_spatial     = False
NS_plot_qq       = False 

# ── Trajectories ──────────────────────────────────────────────────────────
find_hydrographs = False
plot_one         = False

# ── Export ────────────────────────────────────────────────────────────────
export_results       = False
export_signal        = False
apply_threshold_mask = False
discharge_path       = 'daily_flows_wide_by_gauge_name.csv'
parent_dir           = 'Test'
run_name             = 'baseline'

# Load Data

In [8]:
# Verbose command
output_ctx = contextlib.nullcontext() if verbose else suppress_output()

# Pull discharge data
if discharge_source == 'nwis':
    checker = MississippiGaugeChecker()
    res = checker.scan(
        min_years=90,
        completeness=1.0,
        min_end="2020-01-01",       
        allowed_site_no_lengths=(8,),
    )
    print(res.count)
    
    site_meta = res.meta
    df90 = res.df
    if plot_gauges:
        coords = checker.fetch_site_coords(df90["site_no"].astype(str).unique())
        checker.plot(coords, title="Mississippi Basin streamgages with ≥90 years (high continuity)")
else:
    if experiment == 'subset':
        site_meta = {
            "st_louis":     {"site": "07010000", "start": "1861-01-01", "end": "2026-01-01"},
            "kansas_city":  {"site": "06893000", "start": "1929-01-01", "end": "2026-01-01"},
            "minneapolis":  {"site": "05288500", "start": "1932-01-01", "end": "2026-01-01"},
            "clinton":      {"site": "05420500", "start": "1873-01-01", "end": "2026-01-01"},
        }
    else:
        # Load enriched metadata (now includes start / end)
        meta = pd.read_csv(
            "nwis_site_metadata_filtered_da_ge_100sqmi.csv",
            dtype={"site_no": str}
        )

        # Identify the gauge-name column
        if "gauge_name" in meta.columns:
            name_col = "gauge_name"
        elif "station_nm" in meta.columns:
            name_col = "station_nm"
        else:
            raise ValueError("Metadata CSV must have either 'gauge_name' or 'station_nm'.")
        
        # Clean strings
        meta[name_col] = meta[name_col].astype(str).str.strip()
        meta["site_no"] = meta["site_no"].astype(str).str.zfill(8)
        
        # Parse dates
        meta["start"] = pd.to_datetime(meta["start"], errors="coerce")
        meta["end"]   = pd.to_datetime(meta["end"], errors="coerce")
        
        # Build dictionary
        site_meta = {}
        
        for _, row in meta.iterrows():
            gauge_name = row[name_col]
            site_no = row["site_no"]
        
            site_meta[gauge_name] = {
                "site": site_no,
                "start": row["start"].strftime("%Y-%m-%d") if pd.notna(row["start"]) else None,
                "end":   row["end"].strftime("%Y-%m-%d")   if pd.notna(row["end"])   else None,
            }  


# Load climate indices
climate_ind = pd.read_csv('Climatological/climate_indices_no_pna.csv', parse_dates=[0], index_col=0)

gauges = [
    GaugeSpec(name=k, site_no=v["site"], start=v["start"], end=v["end"])
    for k, v in site_meta.items()
]
sites = sorted(list(site_meta.keys()))

In [9]:
# Dataframe construction
ds = HydroClimateDataset(
    gauges=gauges,
    discharge_source=discharge_source,
    discharge_local_path=discharge_path,
    discharge_datetime_col="datetime",
    climate_source='Climatological/climate_indices_no_pna.csv',
    climate_datetime_col=0,
    climate_indices=climate_vars,
    standardize_cols=sites,
    verbose=True,
).build()

DISCHARGE COLS: ['ALLEGHENY RIVER AT SALAMANCA NY', 'CHADAKOIN RIVER AT FALCONER NY', 'Brokenstraw Creek at Youngsville, PA', 'Allegheny River at Franklin, PA', 'Allegheny River at Parker, PA', 'Casselman River at Markleton, PA', 'Laurel Hill Creek at Ursina, PA', 'Ohio River at Sewickley, PA', 'Mahoning River at Pricetown OH', 'Little Beaver Creek near East Liverpool OH', 'Nimishillen Creek at North Industry OH', 'Tuscarawas River at Newcomerstown OH', 'Killbuck Creek at Killbuck OH', 'Hocking River at Enterprise OH', 'NEW RIVER NEAR GALAX, VA', 'GREENBRIER RIVER AT ALDERSON, WV', 'WILLIAMS RIVER AT DYER, WV', 'GAULEY RIVER ABOVE BELVA, WV', 'KANAWHA RIVER AT KANAWHA FALLS, WV', "Scioto River below O'Shaughnessy Dam nr Dublin OH"]
GAUGE NAMES: ['ALLEGHENY RIVER AT SALAMANCA NY', 'CHADAKOIN RIVER AT FALCONER NY', 'Brokenstraw Creek at Youngsville, PA', 'Allegheny River at Franklin, PA', 'Allegheny River at Parker, PA', 'Casselman River at Markleton, PA', 'Laurel Hill Creek at Ursina, P

In [10]:
# Return period thresholds
rl_emp, rl_gum = ds.compute_return_levels(
    cols=sites,
    targets=(2, 5, 10, 100),
    fit_gumbel=True,
)

if std_thres == 'bimonthly':
    thr_map = (
    ds.monthly_max[[f"{site}_max" for site in sites]]
        .mean()
        .rename(lambda c: c.replace("_max", ""))
        .to_dict()
    )

else:
    thr_map = ds.threshold_map_from_rl(rl_emp, 2)

# Threshold exceedance event extraction
marks, events = ds.build_monthly_event_marks(ds.discharge_daily, threshold_map=thr_map)


export_events = historic_events_to_export(events)

if export_results:
    export_events.to_csv(f'{parent_dir}/events_historic_{run_name}.csv')

# Wavelet

In [11]:
wa = WaveletAnalyzer(WaveletConfig(dt=1/12, dj=0.025))

cols = sorted(
    [f"{site}_max_z" for site in sites] +
    [f"{site}_sum_z" for site in sites]
)

if use_wave:
    climate_filtered, sig_scales_all = wa.process_dataframe_columns(
        df=ds.monthly_max,
        cols=cols,
        siglvl=siglvl,
        sigtest=sigtest,
        plot_wave=plot_wave,
        drop_zero_frequency_months=True,
        frequency_col_name="Frequency",
    )
    signal_suffix = "max_z_wave"

else:
    climate_filtered = ds.monthly_max.copy()
    sig_scales_all   = {}

    # Create _wave alias columns so downstream code is unaffected
    for site in sites:
        climate_filtered[f"{site}_max_z_wave"] = climate_filtered[f"{site}_max_z"]
        climate_filtered[f"{site}_sum_z_wave"] = climate_filtered[f"{site}_sum_z"]

    # Compute lag columns — wa.process_dataframe_columns does this internally
    # in the use_wave=True path so we must replicate it here
    bases_wave = (
        [f"{site}_max_z_wave" for site in sites] +
        [f"{site}_sum_z_wave" for site in sites]
    )
    for col in bases_wave:
        climate_filtered[f"{col}_lag"] = climate_filtered[col].shift(1)

    if plot_wave:
        print("plot_wave=True has no effect when use_wave=False.")

    signal_suffix = "max_z_wave"

# Export historic fids (shared by both branches)
export_fids = events_to_export(
    climate_filtered,
    signal_suffix=signal_suffix,
    date_col="Unnamed: 0",
    default_sim_id=0,
    keep_zeros=True,
)
if export_results:
    export_fids.to_csv(f'{parent_dir}/fids_historic_{run_name}.csv')

Processing ALLEGHENY RIVER AT SALAMANCA NY_max_z ...
Red Noise AR1 Coefficient: 0.3167305305980622
Red Noise Sig Test Results
Significant scales (within COI):
[0.21 0.22 0.22 0.27 0.27 0.28 0.28 0.29 0.29 0.36 0.36 0.37 0.38 0.38
 0.39 0.4  0.4  0.41 0.42 0.58 0.59 0.6  0.61 0.62 0.63 0.64 0.66 0.67
 0.68 0.69 0.7  0.71 0.73 0.74 0.75 0.77 0.78 0.79 0.81 0.82 0.84 0.85
 0.86 0.88 0.9  0.91 0.93 0.94 0.96 0.98 0.99 1.01 1.03 1.05 1.06 1.08
 1.1  1.12 1.14 1.16 1.18 1.2  1.22 1.24 1.27 1.29 1.31 1.33 1.36 1.38
 1.4  1.43 1.45 1.48 1.51 1.53 1.56 1.59 1.61 1.64 1.67 1.7  1.73 1.76
 1.79 1.82 1.85 1.89 1.92 1.95 1.99 2.02 2.06 2.09 2.13 2.17 2.2  2.24
 2.28 2.32 2.36 2.4  2.45 2.49 2.53 2.58 2.62 2.67 2.71 2.76 2.81 2.86
 2.91 2.96 3.01 3.06 3.12 3.17 3.23 3.28 3.34 3.4  3.46 3.52 3.58 3.64
 3.71 3.77 3.84 3.9  3.97 4.04 4.11 4.18 4.26 4.33 4.41 4.48 4.56]
Processing ALLEGHENY RIVER AT SALAMANCA NY_sum_z ...
Red Noise AR1 Coefficient: 0.42585052780672505
Red Noise Sig Test Results
Signific

# Transformer

In [12]:
metrics = ["max_z", "sum_z"]
bases   = [f"{site}_{m}_wave" for site in sites for m in metrics]
lags    = [f"{b}_lag" for b in bases]
feature_cols = list(climate_vars) + ["sin_month"] + bases + lags
target_cols  = [f"{site}_max_z_wave" for site in sites]

n_before = climate_filtered.shape[0]
climate_filtered = climate_filtered.dropna(subset=feature_cols)
print(f"Dropped {n_before - climate_filtered.shape[0]} rows.")

Dropped 1 rows.


In [13]:
# --- model ---
exp = HybridKNNTransformer(
    feature_columns=feature_cols,
    target_columns=target_cols,
    seq_length=seq_length,
    pred_horizon=forecast_horizon,
    hidden_dim=hidden_dim,
    model_kind=model_kind,
    num_layers=num_layers,
    nhead=int(hidden_dim/4),
    train_cfg=TrainConfig(lr=lr, batch_size=batch_size, epochs=epochs, patience=patience, early_stop=early_stop),
    mode=mode,   # "forecast_mode" | "analog_mode"
    seed=0,
)

info = exp.fit(climate_filtered, val_frac=val_frac, test_frac=test_frac, loss_name=loss_func)

splits   = exp.time_split(climate_filtered, val_frac=val_frac, test_frac=test_frac)
test_ext = splits["test_ext"]
KMean = int(np.clip(round(np.sqrt(len(exp.mean_store))), 5, 200))

# --- configure embedding sampler ---
knn_mean_cfg = KNNMeanConfig(K=KMean, temperature=temp, alpha=alpha, beta=beta)

ens_cfg = EnsembleConfig(
    n_samples=n_sims,
    seed=(0 if mode == "forecast_mode" else None),
    resid_K=KMean,
    resid_temperature=temp,
)

# --------------------
# POINT (windows + latest)
# --------------------
out_point = exp.predict(
    test_ext,
    kind="windows",
    output="point",
    mean="knn_blend",          # used in forecast_mode; ignored in analog_mode
    knn_mean_cfg=knn_mean_cfg,
)
pred = out_point["pred"]
y    = out_point["y"]

next_point = exp.predict(
    climate_filtered,
    kind="latest",
    output="point",
    mean="knn_blend",
    knn_mean_cfg=knn_mean_cfg,
)["pred"]

# --------------------
# ENSEMBLE (windows + latest)
# --------------------
if test_frac > 0:
    out_ens = exp.predict(
        test_ext,
        kind="windows",
        output="ensemble",
        mean="knn_blend",          # used in forecast_mode; ignored in analog_mode
        knn_mean_cfg=knn_mean_cfg,
        ens_cfg=ens_cfg,
    )
    ens   = out_ens["ens"]     # [E,N,H,D]
    point = out_ens["point"]   # [N,H,D]
    y     = out_ens["y"]       # [N,H,D]

next_out = exp.predict(
    climate_filtered,
    kind="latest",
    output="ensemble",
    mean="knn_blend",
    knn_mean_cfg=knn_mean_cfg,
    ens_cfg=ens_cfg,
)
next_ens   = next_out["ens"]    # [E,H,D]
next_point = next_out["point"]  # [H,D]

next_forecast_df = pd.DataFrame(
    next_point,
    columns=sites,
)

Epoch   0/50 | Train 1.043432 | Val 1.098675
Epoch  10/50 | Train 0.692432 | Val 0.737650
Epoch  20/50 | Train 0.666462 | Val 0.747213
Early stopping triggered.


In [14]:
if plot_transformer:
    # --- Plot test forecasts (point + ensemble PI) ---
    exp.plot_test_forecasts_facets(
        climate_filtered,
        val_frac=val_frac,
        test_frac=test_frac,
        mean="knn_blend",
        knn_mean_cfg=knn_mean_cfg,
        ensemble_cfg=ens_cfg,          # <- this triggers PI shading via predict_windows_ensemble()
        pi_lo=0.10,
        pi_hi=0.90,
        pi_color="lightgray",
        pi_alpha=0.35,
        mode="both",               # "rolling_1step" | "first_trajectory" | "both"
        history_frac=0.6,          # optional (or history_window=...)
        title="Test forecasts: KNN blend + residual ensemble (10–90% PI)",
        show=True,
    )

In [15]:
if assess_pred:
    spread_df = EnsembleAssessment.ensemble_spread_summary(ens, target_cols, horizons=[0,1])
    print(spread_df)
    
    cov = EnsembleAssessment.coverage(ens, y, alpha=0.1)  # [H,D]
    print("Coverage (90% band) mean:", cov.mean())
    
    vr = EnsembleAssessment.variance_ratio(ens, y, point=point)  # [H,D]
    print("Variance ratio mean:", vr.mean())
    
    # QQ plot example
    j = target_cols.index(f"{sites[0]}_max_z_wave")  # pick target
    h = 0
    EnsembleAssessment.qq_plot_from_ensemble(ens, y, target_j=j, horizon_h=h, mode="random_draw")
    plt.show()

# PCA ASSESS

In [16]:
if check_PC:
    
    C1 = "#214B89"   # primary blue
    C2 = "#2b8cbe"   # secondary blue
    C3 = "#1D9E75"   # green

    # --- extract embeddings and their corresponding dates ---
    # mean_store entries: (embedding [H], y_trajectory [pred_horizon, J])
    # We need to recover the date of each window's START month
    
    # Rebuild the training sequence start dates
    # _make_xy uses create_sequences_np which slides a window of seq_length
    # over climate_filtered — window i starts at row i
    df_clean = climate_filtered.dropna(subset=feature_cols).copy()
    
    n_windows   = len(exp.mean_store)
    seq_length  = exp.seq_length
    pred_horizon = exp.pred_horizon
    
    # Date of the START of each window (first month in the input sequence)
    # Window i covers rows [i : i+seq_length], target rows [i+seq_length : i+seq_length+pred_horizon]
    window_start_idx = np.arange(n_windows)
    
    if hasattr(df_clean.index, 'to_timestamp'):
        dates = df_clean.index.to_timestamp()
    else:
        dates = pd.to_datetime(df_clean.index)
    
    window_dates = dates[window_start_idx]   # start date of each window
    # midpoint date (more representative of the window)
    window_mid_dates = dates[
        np.minimum(window_start_idx + seq_length // 2, len(dates) - 1)
    ]
    
    # Stack all embeddings: [N, hidden_dim]
    Z = torch.stack([k for k, _ in exp.mean_store]).numpy()   # [N, H]
    N, H = Z.shape
    print(f"Embedding matrix: {N} windows × {H} dimensions")
    print(f"Date range: {window_dates[0].date()} → {window_dates[-1].date()}")

    # Standardise before PCA (embeddings may have different scales per dimension)
    scaler = StandardScaler()
    Z_scaled = scaler.fit_transform(Z)
    
    pca = PCA()
    Z_pca = pca.fit_transform(Z_scaled)   # [N, H]
    
    cumvar = pca.explained_variance_ratio_.cumsum()
    n_50  = int((cumvar < 0.50).sum()) + 1
    n_80  = int((cumvar < 0.80).sum()) + 1
    n_90  = int((cumvar < 0.90).sum()) + 1
    n_99  = int((cumvar < 0.99).sum()) + 1
    
    print(f"Dimensions for  50% variance: {n_50}")
    print(f"Dimensions for  80% variance: {n_80}")
    print(f"Dimensions for  90% variance: {n_90}")
    print(f"Dimensions for  99% variance: {n_99}")
    print(f"\nTop-10 individual % explained:")
    for i, v in enumerate(pca.explained_variance_ratio_[:10]):
        print(f"  PC{i+1:2d}: {v*100:.1f}%")

    # Align climate indices with window midpoint dates
    # climate_filtered has climate_vars as columns
    n_pcs    = min(10, H)   # examine top 10 PCs
    n_clim   = len(climate_vars)
    
    # Build a DataFrame of PC scores aligned to window midpoint dates
    pc_df = pd.DataFrame(
        Z_pca[:, :n_pcs],
        columns=[f"PC{i+1}" for i in range(n_pcs)],
        index=window_mid_dates,
    )
    
    # Get climate index values at window midpoints
    # Use the midpoint month's climate index value
    clim_at_mid = []
    for d in window_mid_dates:
        # find nearest row in climate_filtered by date
        if hasattr(df_clean.index, 'to_timestamp'):
            idx_dates = df_clean.index.to_timestamp()
        else:
            idx_dates = pd.to_datetime(df_clean.index)
        nearest_idx = np.argmin(np.abs(idx_dates - d))
        clim_at_mid.append(df_clean[climate_vars].iloc[nearest_idx].values)
    
    clim_mid_df = pd.DataFrame(
        clim_at_mid,
        columns=climate_vars,
        index=window_mid_dates,
    )

    # Pearson and Spearman correlations: PC vs climate index
    pearson_r  = np.zeros((n_pcs, n_clim))
    spearman_r = np.zeros((n_pcs, n_clim))
    pearson_p  = np.zeros((n_pcs, n_clim))
    
    for i, pc in enumerate([f"PC{j+1}" for j in range(n_pcs)]):
        for j, cv in enumerate(climate_vars):
            mask = np.isfinite(pc_df[pc].values) & np.isfinite(clim_mid_df[cv].values)
            r, p = pearsonr(pc_df[pc].values[mask], clim_mid_df[cv].values[mask])
            rs, _ = spearmanr(pc_df[pc].values[mask], clim_mid_df[cv].values[mask])
            pearson_r[i, j]  = r
            spearman_r[i, j] = rs
            pearson_p[i, j]  = p
    
    print("Pearson r (PC vs climate index):")
    print(pd.DataFrame(pearson_r,
                       index=[f"PC{i+1}" for i in range(n_pcs)],
                       columns=climate_vars).round(3))

    # Compute pairwise Euclidean distances in original embedding space
    # to assess whether kNN retrieval is well-conditioned
    from sklearn.metrics import pairwise_distances
    
    # Subsample if N is large to keep computation manageable
    max_pts = min(N, 500)
    idx_sub = np.random.default_rng(0).choice(N, size=max_pts, replace=False)
    Z_sub   = Z[idx_sub]
    
    D_mat = pairwise_distances(Z_sub, metric="euclidean")   # [max_pts, max_pts]
    np.fill_diagonal(D_mat, np.nan)
    
    # For each point: distance to nearest neighbour vs distance to K-th neighbour
    K = int(np.clip(round(np.sqrt(N)), 5, 200))
    d_sorted = np.sort(D_mat, axis=1)   # sort each row, ignore NaN diagonal
    d_1st  = d_sorted[:, 0]    # nearest neighbour distance
    d_kth  = d_sorted[:, K-1]  # K-th neighbour distance
    d_mean = np.nanmean(D_mat, axis=1)  # mean distance to all others
    
    # Concentration ratio: d_1st / d_mean — close to 1 means poor discrimination
    concentration = d_1st / d_mean
    
    print(f"K (used in retrieval): {K}")
    print(f"\nDistance statistics (original {H}-dim embedding):")
    print(f"  d(1st neighbour):  mean={d_1st.mean():.2f}, std={d_1st.std():.2f}")
    print(f"  d(K-th neighbour): mean={d_kth.mean():.2f}, std={d_kth.std():.2f}")
    print(f"  d(mean all):       mean={d_mean.mean():.2f}, std={d_mean.std():.2f}")
    print(f"  Concentration (d_1st/d_mean): {concentration.mean():.3f}  "
          f"(1.0 = no discrimination, <0.5 = good discrimination)")
    
    # Repeat in PCA-reduced space (top n_90 components)
    Z_pca_red = Z_pca[:, :n_90]
    Z_sub_red = Z_pca_red[idx_sub]
    D_mat_red = pairwise_distances(Z_sub_red, metric="euclidean")
    np.fill_diagonal(D_mat_red, np.nan)
    d_sorted_red = np.sort(D_mat_red, axis=1)
    d_1st_red    = d_sorted_red[:, 0]
    d_mean_red   = np.nanmean(D_mat_red, axis=1)
    conc_red     = d_1st_red / d_mean_red
    
    print(f"\nDistance statistics (PCA-reduced to {n_90} dims):")
    print(f"  Concentration (d_1st/d_mean): {conc_red.mean():.3f}")
    print(f"  Improvement from dimensionality reduction: "
          f"{(concentration.mean() - conc_red.mean()) / concentration.mean() * 100:.1f}%")

    if plot_PC:
        fig = plt.figure(figsize=(16, 12), constrained_layout=True)
        gs  = gridspec.GridSpec(3, 4, figure=fig)

        panel_iter = iter("abcdefghijkl")
        def plabel(ax):
            ax.text(-0.12, 1.06, f"{next(panel_iter)})",
                    transform=ax.transAxes,
                    fontsize=10, fontweight="bold", va="top")
        
        
        # ── Panel A: variance explained (scree + cumulative) ─────────────────────
        ax = fig.add_subplot(gs[0, :2])
        n_show = min(30, H)
        x      = np.arange(1, n_show + 1)
        
        ax2 = ax.twinx()
        ax.bar(x, pca.explained_variance_ratio_[:n_show] * 100,
               color=C1, alpha=0.7, width=0.7, label="Individual %")
        ax2.plot(x, cumvar[:n_show] * 100,
                 color=C3, linewidth=2, marker="o", markersize=3, label="Cumulative %")
        
        # threshold lines
        for thresh, label in [(80, "80%"), (90, "90%"), (99, "99%")]:
            ax2.axhline(thresh, color="#ccc", linewidth=0.7, linestyle="--")
            ax2.text(n_show + 0.3, thresh, label, fontsize=7, va="center", color="#aaa")
        
        ax.set_xlabel("Principal component", fontsize=9)
        ax.set_ylabel("Variance explained (%)", fontsize=9, color=C1)
        ax2.set_ylabel("Cumulative variance (%)", fontsize=9, color=C3)
        ax2.set_ylim(0, 105)
        ax.set_xlim(0.3, n_show + 0.7)
        
        # annotate key thresholds
        for n_k, label in [(n_90, f"{n_90}D→90%"), (n_80, f"{n_80}D→80%")]:
            ax.axvline(n_k, color="#999", linewidth=0.8, linestyle=":")
            ax.text(n_k + 0.2, ax.get_ylim()[1] * 0.95, label,
                    fontsize=7, color="#666", va="top")
        
        ax.set_title("PCA scree plot — embedding space", fontsize=10)
        lines1, labs1 = ax.get_legend_handles_labels()
        lines2, labs2 = ax2.get_legend_handles_labels()
        ax.legend(lines1 + lines2, labs1 + labs2, fontsize=7, frameon=False, loc="center right")
        plabel(ax)

        
        # ── Panel B: PC scores over time (top 4 PCs) ─────────────────────────────
        ax = fig.add_subplot(gs[0, 2:])
        colors_pc = [C1, C2, C3, "#BA7517"]
        t = pd.to_datetime(window_mid_dates)
        
        for i in range(min(4, n_pcs)):
            pc_norm = (Z_pca[:, i] - Z_pca[:, i].mean()) / (Z_pca[:, i].std() + 1e-9)
            ax.plot(t, pc_norm, linewidth=0.8, alpha=0.8,
                    color=colors_pc[i], label=f"PC{i+1} ({pca.explained_variance_ratio_[i]*100:.1f}%)")
        
        ax.axhline(0, color="#ccc", linewidth=0.5)
        ax.set_xlabel("Window midpoint date", fontsize=9)
        ax.set_ylabel("Normalised PC score", fontsize=9)
        ax.set_title("Top 4 PC scores over time", fontsize=10)
        ax.legend(fontsize=7, frameon=False, ncol=2)
        plabel(ax)
        

        # ── Panels C–F: PC vs climate index heatmap ──────────────────────────────
        ax = fig.add_subplot(gs[1, :2])
        n_pcs_show = min(8, n_pcs)
        
        im = ax.imshow(
            pearson_r[:n_pcs_show, :],
            cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto",
        )
        ax.set_xticks(range(n_clim))
        ax.set_xticklabels([v.upper() for v in climate_vars], fontsize=9)
        ax.set_yticks(range(n_pcs_show))
        ax.set_yticklabels([f"PC{i+1}" for i in range(n_pcs_show)], fontsize=9)
        ax.set_title("Pearson r: PC vs climate index", fontsize=10)
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        
        # annotate significant correlations (p < 0.05)
        for i in range(n_pcs_show):
            for j in range(n_clim):
                if pearson_p[i, j] < 0.05:
                    ax.text(j, i, f"{pearson_r[i,j]:.2f}",
                            ha="center", va="center", fontsize=7,
                            color="white" if abs(pearson_r[i,j]) > 0.4 else "black",
                            fontweight="bold")
                else:
                    ax.text(j, i, f"{pearson_r[i,j]:.2f}",
                            ha="center", va="center", fontsize=6, color="#aaa")
        plabel(ax)


        # ── Panel D: Spearman r heatmap ───────────────────────────────────────────
        ax = fig.add_subplot(gs[1, 2:])
        im2 = ax.imshow(
            spearman_r[:n_pcs_show, :],
            cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto",
        )
        ax.set_xticks(range(n_clim))
        ax.set_xticklabels([v.upper() for v in climate_vars], fontsize=9)
        ax.set_yticks(range(n_pcs_show))
        ax.set_yticklabels([f"PC{i+1}" for i in range(n_pcs_show)], fontsize=9)
        ax.set_title("Spearman r: PC vs climate index", fontsize=10)
        fig.colorbar(im2, ax=ax, fraction=0.046, pad=0.04)
        for i in range(n_pcs_show):
            for j in range(n_clim):
                ax.text(j, i, f"{spearman_r[i,j]:.2f}",
                        ha="center", va="center", fontsize=6,
                        color="white" if abs(spearman_r[i,j]) > 0.4 else "#333")
        plabel(ax)
        
        
        # ── Panel E: distance distribution — original vs reduced ─────────────────
        ax = fig.add_subplot(gs[2, :2])
        bins = np.linspace(0, np.percentile(D_mat[~np.isnan(D_mat)], 99), 60)
        ax.hist(D_mat[~np.isnan(D_mat)].ravel(), bins=bins,
                color=C1, alpha=0.6, density=True,
                label=f"Original ({H}D)")
        bins_red = np.linspace(0, np.percentile(D_mat_red[~np.isnan(D_mat_red)], 99), 60)
        ax.hist(D_mat_red[~np.isnan(D_mat_red)].ravel(), bins=bins_red,
                color=C3, alpha=0.6, density=True,
                label=f"PCA-reduced ({n_90}D)")
        ax.set_xlabel("Pairwise Euclidean distance", fontsize=9)
        ax.set_ylabel("Density", fontsize=9)
        ax.set_title("Distance distribution: original vs PCA-reduced", fontsize=10)
        ax.legend(fontsize=8, frameon=False)
        plabel(ax)
        
        
        # ── Panel F: concentration ratio ─────────────────────────────────────────
        ax = fig.add_subplot(gs[2, 2:])
        ax.hist(concentration, bins=40, color=C1, alpha=0.7, density=True,
                label=f"Original ({H}D)  mean={concentration.mean():.3f}")
        ax.hist(conc_red, bins=40, color=C3, alpha=0.7, density=True,
                label=f"PCA-reduced ({n_90}D)  mean={conc_red.mean():.3f}")
        ax.axvline(0.5, color="#A32D2D", linewidth=1, linestyle="--",
                   label="Discrimination threshold (0.5)")
        ax.set_xlabel("d(1st neighbour) / d(mean)", fontsize=9)
        ax.set_ylabel("Density", fontsize=9)
        ax.set_title("kNN discrimination: concentration ratio", fontsize=10)
        ax.legend(fontsize=7.5, frameon=False)
        plabel(ax)
        
        
        fig.suptitle(
            f"Transformer embedding space analysis  "
            f"(N={N} windows, hidden_dim={H})",
            fontsize=11, y=1.01
        )
        plt.savefig("embedding_pca_analysis.png", dpi=150, bbox_inches="tight")
        plt.show()
        print("Saved: embedding_pca_analysis.png")

    print("=" * 55)
    print("EMBEDDING DIMENSION ASSESSMENT")
    print("=" * 55)
    print(f"\nCurrent hidden_dim:          {H}")
    print(f"Training windows (N):        {N}")
    print(f"N / hidden_dim ratio:        {N/H:.1f}  (target: >10)")
    print(f"\nIntrinsic dimensionality:")
    print(f"  Dims for 80% variance:     {n_80}")
    print(f"  Dims for 90% variance:     {n_90}")
    print(f"  Dims for 99% variance:     {n_99}")
    print(f"\nkNN discrimination:")
    print(f"  Concentration ({H}D):    {concentration.mean():.3f}")
    print(f"  Concentration ({n_90}D): {conc_red.mean():.3f}")
    conc_improvement = (concentration.mean() - conc_red.mean()) / concentration.mean() * 100
    print(f"  Improvement from reduction: {conc_improvement:.1f}%")

    # Recommendation
    print(f"\nCLIMATE INDEX CORRELATIONS (significant PCs):")
    for i in range(min(6, n_pcs)):
        top_j = np.argmax(np.abs(pearson_r[i]))
        r_val = pearson_r[i, top_j]
        p_val = pearson_p[i, top_j]
        sig   = "*" if p_val < 0.05 else ""
        print(f"  PC{i+1} ({pca.explained_variance_ratio_[i]*100:.1f}%): "
              f"strongest with {climate_vars[top_j].upper()} "
              f"r={r_val:.3f}{sig}")
    
    print(f"\nRECOMMENDATION:")
    # Rule: use smallest power of 2 >= n_90, capped at 64
    rec_dim = 2 ** int(np.ceil(np.log2(max(n_90, 8))))
    rec_dim = min(rec_dim, 64)
    if concentration.mean() > 0.7:
        print(f"  Concentration ratio {concentration.mean():.3f} > 0.7 — embedding is "
              f"poorly discriminative at hidden_dim={H}.")
        print(f"  Recommend reducing hidden_dim to {rec_dim} "
              f"(covers {n_90}D intrinsic structure with margin).")
    elif concentration.mean() > 0.5:
        print(f"  Concentration ratio {concentration.mean():.3f} is borderline.")
        print(f"  Consider testing hidden_dim={rec_dim} for improved retrieval.")
    else:
        print(f"  Concentration ratio {concentration.mean():.3f} < 0.5 — "
              f"kNN retrieval is reasonably discriminative at hidden_dim={H}.")
        print(f"  Current dimension is appropriate.")
    print("=" * 55)

    # Test PC correlation with seasonal covariates
    sin_at_mid = []
    cos_at_mid = []
    
    for d in window_mid_dates:
        if hasattr(df_clean.index, 'to_timestamp'):
            idx_dates = df_clean.index.to_timestamp()
        else:
            idx_dates = pd.to_datetime(df_clean.index)
        nearest_idx = np.argmin(np.abs(idx_dates - d))
    
        # sin_month is already in climate_filtered
        sin_val = float(df_clean["sin_month"].iloc[nearest_idx])
        # reconstruct cos_month from the month number
        month_num = d.month
        cos_val   = np.cos(2 * np.pi * month_num / 12)
    
        sin_at_mid.append(sin_val)
        cos_at_mid.append(cos_val)
    
    sin_arr = np.array(sin_at_mid)
    cos_arr = np.array(cos_at_mid)
    
    print("PC correlations with seasonal covariates:")
    print(f"{'PC':>4}  {'r(sin)':>8}  {'r(cos)':>8}  {'r(combined)':>12}")
    for i in range(8):
        r_sin, _ = pearsonr(Z_pca[:, i], sin_arr)
        r_cos, _ = pearsonr(Z_pca[:, i], cos_arr)
        # combined seasonal signal = sqrt(r_sin^2 + r_cos^2)
        r_combined = np.sqrt(r_sin**2 + r_cos**2)
        print(f"PC{i+1:>2}  {r_sin:>8.3f}  {r_cos:>8.3f}  {r_combined:>12.3f}")

    signal_cols = [f"{site}_max_z_wave" for site in sites
               if f"{site}_max_z_wave" in df_clean.columns]
    basin_mean  = df_clean[signal_cols].mean(axis=1)
    
    basin_at_mid = []
    for d in window_mid_dates:
        if hasattr(df_clean.index, 'to_timestamp'):
            idx_dates = df_clean.index.to_timestamp()
        else:
            idx_dates = pd.to_datetime(df_clean.index)
        nearest_idx = np.argmin(np.abs(idx_dates - d))
        basin_at_mid.append(float(basin_mean.iloc[nearest_idx]))
    
    basin_arr = np.array(basin_at_mid)
    for i in range(6):
        r, p = pearsonr(Z_pca[:, i], basin_arr)
        print(f"PC{i+1} vs basin-mean signal: r={r:.3f} p={p:.4f}")

    # Correlate PC1 with each site's deseasoned signal
    # to identify the spatial pattern it represents
    signal_cols = [f"{site}_max_z_wave" for site in sites
                   if f"{site}_max_z_wave" in df_clean.columns]
    
    site_r = {}
    for col in signal_cols:
        site_signal = df_clean[col]
        site_at_mid = []
        for d in window_mid_dates:
            if hasattr(df_clean.index, 'to_timestamp'):
                idx_dates = df_clean.index.to_timestamp()
            else:
                idx_dates = pd.to_datetime(df_clean.index)
            nearest_idx = np.argmin(np.abs(idx_dates - d))
            site_at_mid.append(float(site_signal.iloc[nearest_idx]))
    
        r, p = pearsonr(Z_pca[:, 0], np.array(site_at_mid))
        site_name = col.replace("_max_z_wave", "")
        site_r[site_name] = {"r": r, "p": p}
    
    # Top positive and negative loadings
    df_site_r = pd.DataFrame(site_r).T.sort_values("r")
    print("Top 10 negative PC1 loadings (sites PC1 is inversely related to):")
    print(df_site_r.head(10)[["r","p"]].round(3))
    print("\nTop 10 positive PC1 loadings:")
    print(df_site_r.tail(10)[["r","p"]].round(3))

# Analog Mode

In [17]:
# Always build JA store — provides analog_payloads as rank template for
ja_cfg = JointAnalogConfig(
    K=joint_K if joint_K is not None else int(np.clip(round(np.sqrt(len(exp.mean_store))), 5, 200)),
    temperature=joint_temp,
    flood_vars=["_frequency", "_max_intensity", "_duration_days"],
    signal_suffix="_max_z_wave",
    seed=joint_seed,
)
# --- Build joint store ---
ja_sampler = JointAnalogSampler(exp, sites, cfg=ja_cfg)
ja_sampler.build_joint_store(climate_filtered)

# --- Sample ensemble ---
resampled_all_ja, next_ens, analog_payloads = ja_sampler.sample_ensemble(
    climate_filtered,
    n_samples=n_sims,
    return_signal_ensemble=True,
    return_payloads=True,
)

if simulation_mode == "joint-analog":
    resampled_all = resampled_all_ja
    # next_forecast_df: point summary (mean across ensemble) for downstream use
    next_forecast_df = pd.DataFrame(
        next_ens.mean(axis=0),
        columns=sites,
    )
    print(f"Joint analog ensemble: {resampled_all.shape}, next_ens: {next_ens.shape}")

print(f"Analog payloads: {analog_payloads.shape}")

Joint store built: 863 windows, K=28, joint_cols=468 (117 signal + 351 flood)
Analog payloads: (1000, 60, 468)


/home/jovyan/4_Multisite/joint_analog2.py:474: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{site}_forecast_signal"] = df[sc].values
/home/jovyan/4_Multisite/joint_analog2.py:474: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{site}_forecast_signal"] = df[sc].values
/home/jovyan/4_Multisite/joint_analog2.py:474: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) i

In [18]:
if simulation_mode == "joint-analog":

    # ── Build expanded from resampled_all ─────────────────────────────────────
    freq_cols = [f"{s}_frequency" for s in sites]

    n_events_global = (
        resampled_all[freq_cols]
        .max(axis=1)
        .fillna(0)
        .astype(int)
        .clip(lower=0)
    )

    has_events = n_events_global > 0
    base       = resampled_all.loc[has_events].copy()
    expanded   = base.loc[base.index.repeat(n_events_global[has_events])].copy()

    expanded["event_id"] = (
        expanded
        .groupby(level=["sim_id", "t"])
        .cumcount()
        .to_numpy() + 1
    )

    expanded = (
        expanded
        .reset_index()
        .set_index(["sim_id", "t", "event_id"])
        .sort_index()
        .copy()
    )

    print(f"expanded: {expanded.shape}  index=(sim_id, t, event_id)")

    # sanity check — catch missing columns before assemble_and_export
    required = (
        [f"{s}_frequency"       for s in sites] +
        [f"{s}_max_intensity"   for s in sites] +
        [f"{s}_duration_days"   for s in sites] +
        [f"{s}_forecast_signal" for s in sites]
    )
    missing_cols = [c for c in required if c not in expanded.columns]
    if missing_cols:
        raise ValueError(
            f"expanded is missing columns expected by assemble_and_export:\n"
            f"  {missing_cols}\nAvailable: {sorted(expanded.columns)}"
        )

# NS Process Mode

## Fit Copula

In [19]:
if simulation_mode == "copula":
    # First fit to max intensity by month
    copula_sims = {}
    copula_simulators = {}          # store fitted simulators for direct sampling
    original_n = len(climate_filtered)  # the number of months used per-site
    
    for var in sites:
        VAR_COLS = [f"{var}_frequency", f"{var}_max_z_wave"]
    
        specs = {
        f"{var}_frequency": {"name": frequency_dist},
        f"{var}_max_z_wave": {"name": signal_dist},
        }
        
        sim_param = EmpiricalCopulaSimulator(
            climate_filtered[VAR_COLS],
            seed=789
        )
        
        sim_param.fit_empirical_copula(method="ordinal", smooth_ranks=True)
        marginals = {k: make_marginal(v) for k, v in specs.items()}
        sim_param.fit_marginals(marginals=marginals, run_ks=True)
        
        samp_param = sim_param.sample(
            n_samples=original_n,
            bootstrap=True,
            enforce_nonneg_int_cols=[f"{var}_frequency", f"{var}_duration_days"], 
            use_u=True
        )
    
        copula_sims[var] = samp_param
        copula_simulators[var] = sim_param

        if plot_marginals:
            sim_param.plot_marginals(simulated=samp_param[VAR_COLS], which="both", bins=60, figsize=(12, 8))

In [20]:
if simulation_mode == "copula":
    multi_copula_sims = {}
    multi_copula_simulators = {}
    multi_copula_fallback = {}   # track which sites fell back to primary copula

    MIN_MULTI_SAMPLES = 30       # minimum observations to fit a reliable multi-event copula

    freq_cols = [c for c in climate_filtered.columns if c.endswith("_frequency")]

    for var in sites:
        MULTI_VAR_COLS = [f"{var}_duration_days", f"{var}_max_intensity", f"{var}_max_z_wave"]

        multi_specs = {
            f"{var}_duration_days": {"name": duration_dist},
            f"{var}_max_intensity": {"name": intensity_dist},
            f"{var}_max_z_wave":    {"name": signal_dist},
        }

        climate_filtered_multi_fit = climate_filtered[climate_filtered[f"{var}_frequency"] > 1]
        n_multi = len(climate_filtered_multi_fit)

        if n_multi < MIN_MULTI_SAMPLES:
            # Insufficient data — fall back to primary copula for this site.
            # sample_events() will draw duration/intensity from the primary
            # copula pool which was fit on all months, giving a less specific
            # but statistically reliable draw.
            print(f"  [{var}] only {n_multi} multi-event months — falling back to primary copula")
            multi_copula_fallback[var] = True

            # Reuse primary copula sim/simulator — already contains
            # duration_days and max_intensity if they were included,
            # otherwise draw from marginals fitted on all months below.
            # Here we fit marginals only (no copula) on the full record.
            FALLBACK_COLS = [f"{var}_duration_days", f"{var}_max_intensity", f"{var}_max_z_wave"]
            fallback_data = climate_filtered[
                (climate_filtered[f"{var}_frequency"] > 0) &
                climate_filtered[FALLBACK_COLS].notna().all(axis=1)
            ][FALLBACK_COLS]

            if len(fallback_data) >= 10:
                sim_fallback = EmpiricalCopulaSimulator(fallback_data, seed=789)
                sim_fallback.fit_empirical_copula(method="ordinal", smooth_ranks=True)
                marginals_fallback = {k: make_marginal(v) for k, v in multi_specs.items()}
                sim_fallback.fit_marginals(marginals=marginals_fallback, run_ks=True)
                samp_fallback = sim_fallback.sample(
                    n_samples=original_n,
                    bootstrap=True,
                    use_u=True,
                )
                multi_copula_sims[var]       = samp_fallback
                multi_copula_simulators[var] = sim_fallback
            else:
                # Site has almost no event data at all — skip entirely.
                # sample_events() will produce no rows for this site,
                # and expanded will fill duration/intensity with 0.
                print(f"  [{var}] insufficient event data even for fallback — skipping")
            continue

        multi_copula_fallback[var] = False

        sim_param_multi = EmpiricalCopulaSimulator(
            climate_filtered_multi_fit[MULTI_VAR_COLS],
            seed=789
        )
        sim_param_multi.fit_empirical_copula(method="ordinal", smooth_ranks=True)
        marginals_multi = {k: make_marginal(v) for k, v in multi_specs.items()}
        sim_param_multi.fit_marginals(marginals=marginals_multi, run_ks=True)

        samp_param_multi = sim_param_multi.sample(
            n_samples=original_n,
            bootstrap=True,
            enforce_nonneg_int_cols=[f"{var}_mean_duration_nonmax"],
            use_u=True,
        )

        multi_copula_sims[var]       = samp_param_multi
        multi_copula_simulators[var] = sim_param_multi

        if plot_marginals:
            sim_param_multi.plot_marginals(
                simulated=samp_param_multi[MULTI_VAR_COLS], which="both",
                bins=60, figsize=(12, 8)
            )

    n_fallback = sum(multi_copula_fallback.values())
    print(f"Multi-event copula: {len(sites) - n_fallback}/{len(sites)} sites fit, "
          f"{n_fallback} fell back to primary copula.")

  [BIG BLACK RIVER NR BOVINA, MS] only 24 multi-event months — falling back to primary copula
  [JAMES RIVER NEAR SCOTLAND,SD] only 9 multi-event months — falling back to primary copula
  [KANKAKEE RIVER AT SHELBY, IN] only 21 multi-event months — falling back to primary copula
  [MISSISSIPPI RIVER AT HWY 610 IN BROOKLYN PARK, MN] only 19 multi-event months — falling back to primary copula
  [MISSISSIPPI RIVER AT WINONA, MN] only 16 multi-event months — falling back to primary copula
  [PATOKA RIVER NEAR PRINCETON, IN] only 15 multi-event months — falling back to primary copula
  [WABASH RIVER AT MT. CARMEL, IL] only 26 multi-event months — falling back to primary copula
Multi-event copula: 110/117 sites fit, 7 fell back to primary copula.


In [21]:
if simulation_mode == "copula" and cop_plot_qq:
    for site in sites:
        n_quantiles=200
        quantile_clip=(0.001, 0.999)
        x_obs = climate_filtered[f'{site}_max_z_wave'].dropna().to_numpy()
        x_sim = copula_sims[site][f'{site}_max_z_wave'].dropna().to_numpy()
        
        qs = np.linspace(quantile_clip[0], quantile_clip[1], n_quantiles)
        
        q_obs = np.quantile(x_obs, qs)
        q_sim = np.quantile(x_sim, qs)
        
        plt.scatter(q_obs, q_sim, s=12, alpha=0.7)
        lo = min(q_obs.min(), q_sim.min())
        hi = max(q_obs.max(), q_sim.max())
        plt.title("Signal")
        plt.plot([lo, hi], [lo, hi], "k--", lw=1)
    plt.show()

    for site in sites:
        n_quantiles=200
        quantile_clip=(0.001, 0.999)
        x_obs = climate_filtered[f'{site}_frequency'].dropna().to_numpy()
        x_sim = copula_sims[site][f'{site}_frequency'].dropna().to_numpy()
        
        qs = np.linspace(quantile_clip[0], quantile_clip[1], n_quantiles)
        
        q_obs = np.quantile(x_obs, qs)
        q_sim = np.quantile(x_sim, qs)
        
        plt.scatter(q_obs, q_sim, s=12, alpha=0.7)
        lo = min(q_obs.min(), q_sim.min())
        hi = max(q_obs.max(), q_sim.max())
        plt.title("Frequency")
        plt.plot([lo, hi], [lo, hi], "k--", lw=1)
    plt.show()

    for site in sites:
        n_quantiles = 200
        quantile_clip = (0.001, 0.999)
    
        # Filter months with at least one event
        cf = climate_filtered[climate_filtered[f"{var}_frequency"] > 0]
    
        x_obs = cf[f'{site}_max_intensity'].dropna().to_numpy()
        x_sim = multi_copula_sims[site][f'{site}_max_intensity'].dropna().to_numpy()
    
        # --- Standard normalization ---
        x_obs = z(x_obs)
        x_sim = z(x_sim)
        # --------------------------------
    
        qs = np.linspace(quantile_clip[0], quantile_clip[1], n_quantiles)
        q_obs = np.quantile(x_obs, qs)
        q_sim = np.quantile(x_sim, qs)
    
        plt.scatter(q_obs, q_sim, s=12, alpha=0.7)
    
        lo = min(q_obs.min(), q_sim.min())
        hi = max(q_obs.max(), q_sim.max())
        plt.title("Intensity")
        plt.plot([lo, hi], [lo, hi], "k--", lw=1)
    plt.show()

    for site in sites:
        n_quantiles = 200
        quantile_clip = (0.001, 0.999)
    
        # Filter months with at least one event
        cf = climate_filtered[climate_filtered[f"{var}_frequency"] > 0]
    
        x_obs = cf[f'{site}_duration_days'].dropna().to_numpy()
        x_sim = multi_copula_sims[site][f'{site}_duration_days'].dropna().to_numpy()
    
        # --- Standard normalization ---
        x_obs = z(x_obs)
        x_sim = z(x_sim)
        # --------------------------------
    
        qs = np.linspace(quantile_clip[0], quantile_clip[1], n_quantiles)
        q_obs = np.quantile(x_obs, qs)
        q_sim = np.quantile(x_sim, qs)
    
        plt.scatter(q_obs, q_sim, s=12, alpha=0.7)
    
        lo = min(q_obs.min(), q_sim.min())
        hi = max(q_obs.max(), q_sim.max())
        plt.title("Duration")
        plt.plot([lo, hi], [lo, hi], "k--", lw=1)
    plt.show()

## NS Sampler

In [22]:
if simulation_mode == "copula":
    # ens: [n_ens, T, S] where S == len(sites)
    n_ens, T, S = next_ens.shape
    assert S == len(sites)
    
    signal_col_by_site = {s: f"{s}_max_z_wave" for s in sites}

In [23]:
if simulation_mode == "copula":
    aq_sampler = AnalogQuantileSampler(
        copula_sims            = copula_sims,
        copula_simulators      = copula_simulators,
        sites                  = sites,
        primary_vars           = ["_frequency", "_max_z_wave"],
        multi_copula_sims      = multi_copula_sims,
        multi_copula_simulators= multi_copula_simulators,
        multi_vars             = ["_duration_days", "_max_intensity", "_max_z_wave"],
        signal_suffix          = "_max_z_wave",
        jitter                 = aq_std,
        seed                   = 42,
    )
    print("AnalogQuantileSampler ready.")

AnalogQuantileSampler ready.


In [24]:
if simulation_mode == "copula":
    resampled_all = aq_sampler.sample(
        analog_payloads    = analog_payloads,      # (n_sims, T, n_joint_cols)
        next_ens           = next_ens,             # (n_sims, T, n_sites)
        analog_joint_cols  = ja_sampler.joint_cols,# column ordering from JA store
    )
    print(f"resampled_all: {resampled_all.shape}")

resampled_all: (60000, 351)


In [25]:
if simulation_mode == "copula":
    event_df = aq_sampler.sample_events(
        resampled_all      = resampled_all,
        analog_payloads    = analog_payloads,
        analog_joint_cols  = ja_sampler.joint_cols,
    )
    print(f"event_df: {event_df.shape}")

event_df: (4539331, 351)


In [26]:
if simulation_mode == "copula":

    # ── Build expanded from resampled_all ─────────────────────────────────────
    freq_cols = [f"{s}_frequency" for s in sites]

    n_events_global = (
        resampled_all[freq_cols]
        .max(axis=1)
        .fillna(0)
        .astype(int)
        .clip(lower=0)
    )

    has_events = n_events_global > 0
    base       = resampled_all.loc[has_events].copy()
    expanded   = base.loc[base.index.repeat(n_events_global[has_events])].copy()

    expanded["event_id"] = (
        expanded
        .groupby(level=["sim_id", "t"])
        .cumcount()
        .to_numpy() + 1
    )

    expanded = (
        expanded
        .reset_index()
        .set_index(["sim_id", "t", "event_id"])
        .sort_index()
        .copy()  # defragment
    )

    # ensure duration/intensity columns exist and start at 0
    suffixes = ["_duration_days", "_max_intensity"]
    needed   = [f"{s}{suffix}" for s in sites for suffix in suffixes]
    missing  = [c for c in needed if c not in expanded.columns]

    if missing:
        defaults = pd.DataFrame(0.0, index=expanded.index, columns=missing)
        expanded = pd.concat([expanded, defaults], axis=1)

    if not event_df.empty:
        for s in sites:
            try:
                site_events = event_df.xs(s, level="site")
            except KeyError:
                continue
            site_cols = [c for c in site_events.columns if c.startswith(f"{s}_")]
            expanded.loc[site_events.index, site_cols] = site_events[site_cols].to_numpy()

# Export

In [27]:
# ── Shared post-simulation assembly (both modes) ──────────────────────────────
events_export, monthly_sim = assemble_and_export(
    expanded        = expanded,
    resampled_all   = resampled_all,
    ds              = ds,
    sites           = sites,
    thr_map         = thr_map,
    parent_dir      = parent_dir,
    run_name        = run_name,
    apply_threshold_mask = apply_threshold_mask,
    export_results       = export_results,
)

In [28]:
if export_signal:
    # next_ens: shape = (n_sims, T_sim, S)
    n_sims, T_sim, S = next_ens.shape
    assert S == len(sites), "Site dimension of next_ens must match len(sites)"
    
    # MultiIndex over (Sim, t)
    sim_index = pd.MultiIndex.from_product(
        [range(n_sims), range(T_sim)],
        names=["Sim", "t"]
    )
    
    # Flatten to (n_sims * T_sim, S)
    sim_flat = next_ens.reshape(-1, S)
    
    sim_df = pd.DataFrame(sim_flat, index=sim_index, columns=sites)

    sim_df.to_csv(f"{parent_dir}/signal_{run_name}.csv", index=False)

# Plots

In [29]:
if plot_spatial:
    suffixes = ["_max_intensity", "_frequency", "_duration_days", "_max_z_wave"]

    corr_results, tail_results = compute_corr_results(
        resampled_all    = expanded,
        climate_filtered = climate_filtered,
        sites            = sites,
        suffixes         = suffixes,
        ja_sampler       = ja_sampler if simulation_mode == "joint-analog" else None,
    )

    plot_spatial_correlations(corr_results, tail_results, run_name)

In [30]:
if NS_plot_qq:
    obs_signal = {
        site: export_fids.loc[export_fids["Site"] == site, "Signal"].dropna().to_numpy()
        for site in sites
    }
    
    sim_signal = {
        site: monthly_sim.loc[monthly_sim["Site"] == site, "Signal"].dropna().to_numpy()
        for site in sites
    }

    if experiment == 'subset':
        plot_qq_multisite("Signal", obs_signal, sim_signal, sites)
    else:
        plot_qq_overlay("Signal", obs_signal, sim_signal, sites)

    obs_intensity = {
        site: export_events.loc[
            (export_events["Site"] == site) &
            (export_events["Frequency"] > 0) &
            (export_events["Intensity"] > 0),
            "Intensity"
        ].to_numpy()
        for site in sites
    }
    
    sim_intensity = {
        site: events_export.loc[
            (events_export["Site"] == site) &
            (events_export["Frequency"] > 0) &
            (events_export["Intensity"] > 0),
            "Intensity"
        ].to_numpy()
        for site in sites
    }

    if experiment == 'subset':
        plot_qq_multisite("Intensity", obs_intensity, sim_intensity, sites)
    else:
        plot_qq_overlay("Intensity", obs_intensity, sim_intensity, sites)

    obs_duration = {
        site: export_events.loc[
            (export_events["Site"] == site) &
            (export_events["Frequency"] > 0) &
            (export_events["Duration"] > 0),
            "Duration"
        ].to_numpy()
        for site in sites
    }
    
    sim_duration = {
        site: events_export.loc[
            (events_export["Site"] == site) &
            (events_export["Frequency"] > 0) &
            (events_export["Duration"] > 0),
            "Duration"
        ].to_numpy()
        for site in sites
    }
    
    if experiment == 'subset':
        plot_qq_multisite("Duration", obs_duration, sim_duration, sites)
    else:
        plot_qq_overlay("Duration", obs_duration, sim_duration, sites)
    
    obs_freq = {
        site: export_fids.loc[export_fids["Site"] == site, "Frequency"].dropna().to_numpy()
        for site in sites
    }
    
    sim_freq = {
        site: monthly_sim.loc[monthly_sim["Site"] == site, "Frequency"].dropna().to_numpy()
        for site in sites
    }
    
    if experiment == 'subset':
        plot_qq_multisite("Frequency", obs_freq, sim_freq, sites)
    else:
        plot_qq_overlay("Frequency", obs_freq, sim_freq, sites)

# Trajectory Bootstrapping

In [31]:
if find_hydrographs:
    cfg = SubdailyRefitterConfig(verbose=True, duration_metric="logabs")
    refitter = SubdailyHydrographRefitter(
        site_meta,
        events_df=events,
        fetch_iv_window=fetch_iv_window,
        config=cfg
    )
    
    events_iv = refitter.prefilter_events_for_iv(
        search_start="1860-01-01",
        search_end="2026-01-01",
        step_years=10,
        refine=True,
        keep_unknown=False,
    )
    
    refitter.events_df = events_iv
    refitter.build_library()

    subdaily_sim = refitter.refit_simulations(
        filtered_df=expanded,
        id_col=None,
    )

    if plot_one:
        example_id = subdaily_sim["sim_event_id"].iloc[0] # Eg. pick the first event id
        plot_sim_hydrograph(subdaily_sim, example_id)

    if export_results:
        subdaily_sim.to_csv(f'{parent_dir}/hydrographs_simulated_{run_name}.csv')